In [1]:
import pandas as pd
import numpy as np
from keras.layers import Dense,SimpleRNN,Embedding
from keras.models import Sequential
from tensorflow.keras.utils import to_categorical

In [2]:
df=pd.read_csv(r"C:\Users\DELL\Downloads\fake_job_postings.csv (1).xls")

C:\Users\DELL\AppData\Local\Temp\ipykernel_4068\3630556540.py:1: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(r"C:\Users\DELL\Downloads\fake_job_postings.csv (1).xls")


In [3]:
df['title']=df['title'].apply(str)
df['location']=df['location'].apply(str)
df['department']=df['department'].apply(str)
df['salary_range']=df['salary_range'].apply(str)
df['company_profile']=df['company_profile'].apply(str)
df['description']=df['description'].apply(str)
df['requirements']=df['requirements'].apply(str)
df['benefits']=df['benefits'].apply(str)
df['has_company_logo']=df['has_company_logo'].apply(str)
df['employment_type']=df['employment_type'].apply(str)
df['required_experience']=df['required_experience'].apply(str)
df['industry']=df['industry'].apply(str)

In [4]:
df['X'] = df['title']+ " " + df['location']+ " " + df['department']+ " " + df['salary_range']+ " " + df['company_profile']+ " " + df['description']+ " " + df['requirements']+ " " + df['benefits']+ " " + df['has_company_logo']+ " " + df['employment_type']+ " " + df['required_experience']+ " " + df['industry']

In [5]:
df['Y']=df['fraudulent']

In [6]:
df=df.loc[:,['X','Y']]

In [7]:
from sklearn.model_selection import train_test_split
train , test = train_test_split( df , test_size = .2)

df1=df[df.Y==1]
train=pd.concat([train,df1,df1,df1,df1,df1,df1,df1])

In [8]:
train.Y.value_counts()

Y
0    13604
1     6762
Name: count, dtype: int64

In [9]:
train_x = train.iloc[: , 0]
train_y = train.iloc[: , 1]

test_x = test.iloc[: , 0 ]
test_y = test.iloc[: , 1 ]

In [10]:
train_y = to_categorical(train_y)

In [11]:
max_num_words = 10000  # HOW MANY WORDS DO WE CONSIDER FROM THE ENTIRE CORPUS
seq_len = 50           # HOW MANY WORDS DO WE CONSIDER IN EACH DOCUMENT / MESSAGE 
embedding_size = 100   # VECTOR LENGTH OF EACH WORD

from tensorflow.keras.preprocessing.text import Tokenizer # ASSIGN A NUMBER TO EACH WORD
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [12]:
tokenizer = Tokenizer(num_words = max_num_words )
tokenizer.fit_on_texts(df.X)
train_x = tokenizer.texts_to_sequences(train_x)

In [13]:
train_x = pad_sequences(train_x , maxlen = seq_len )

In [14]:
test_x = tokenizer.texts_to_sequences( test_x )
test_x = pad_sequences(test_x , maxlen = seq_len )

In [15]:
model = Sequential() 
model.add(Embedding(input_dim = max_num_words , input_length = seq_len , output_dim = embedding_size ))

model.add(SimpleRNN(32))

model.add(Dense( 2 , activation = 'softmax' )) # O/P THIS 2 BCOZ U HAVE 2 LEVEL IN TARGET VARIABLE 
from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)
model.compile(optimizer = 'adam' , loss = 'categorical_crossentropy' ,metrics = ['accuracy'] )

C:\Users\DELL\anaconda3\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [17]:
model.fit(train_x , train_y , epochs = 5 , batch_size = 100 , validation_split = .2 )

Epoch 1/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.9999 - loss: 0.0018 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 2/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 1.0000 - loss: 8.2828e-04 - val_accuracy: 1.0000 - val_loss: 0.0018
Epoch 3/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 1.0000 - loss: 5.2628e-04 - val_accuracy: 1.0000 - val_loss: 9.3851e-04
Epoch 4/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 1.0000 - loss: 3.1415e-04 - val_accuracy: 1.0000 - val_loss: 6.5793e-04
Epoch 5/5
163/163 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 1.0000 - loss: 2.3288e-04 - val_accuracy: 1.0000 - val_loss: 4.8210e-04


In [18]:
pred = model.predict(test_x)
from sklearn.metrics import confusion_matrix
pred_cate = pred.argmax(axis = 1 )
tab = confusion_matrix(test_y , pred_cate)
tab.diagonal().sum() / tab.sum() * 100

112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


99.32885906040269

In [19]:
from sklearn.metrics import confusion_matrix
tab = confusion_matrix(test_y , pred_cate)
tab

array([[3386,   24],
       [   0,  166]], dtype=int64)

In [20]:
import os
import pickle

# 1. Save the trained model
model.save(r"C:\Users\DELL\OneDrive\Desktop\projects\sentiment_lstm_model.keras")

# 2. Save the tokenizer
with open(r"C:\Users\DELL\OneDrive\Desktop\projects\tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# 3. Confirm both files exist
print("Model saved:", os.path.exists("sentiment_lstm_model.keras"))
print("Tokenizer saved:", os.path.exists("tokenizer.pkl"))

Model saved: True
Tokenizer saved: True
